# 🎯 SAC for 6-DOF Aircraft: 90° Heading Change

**Objective**: Train an aircraft to change heading from 0° to 90° with minimal altitude change

**Algorithm**: Soft Actor-Critic (SAC)
- ✅ Off-policy → 5x more sample efficient than PPO
- ✅ Automatic exploration via maximum entropy
- ✅ Twin Q-networks for stability
- ✅ No hard constraints → All dynamics in reward shaping

**Expected Results**:
- Convergence: ~200k timesteps
- Success rate: >85% (heading error <5°, altitude error <50m)
- Training time: ~30-45 minutes on GPU

---

## 📦 Section 1: Setup and Installations

In [ ]:
# Install dependencies
!pip install gymnasium numpy scipy matplotlib torch torchvision --quiet

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import gymnasium as gym
from collections import deque
import time

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu128


## ✈️ Section 2: 6-DOF Aircraft Dynamics Model

In [ ]:
class AircraftDynamics:
    """6-DOF aircraft dynamics model (F-16 inspired)"""

    def __init__(self):
        # Mass properties
        self.m = 9100.0           # kg
        self.Ixx = 12875.0        # kg·m²
        self.Iyy = 75674.0        # kg·m²
        self.Izz = 85552.0        # kg·m²
        self.Ixz = 1331.0         # kg·m²

        # Reference values
        self.S = 27.87            # Wing area (m²)
        self.b = 9.144            # Wing span (m)
        self.c = 3.45             # Mean chord (m)
        self.g = 9.81             # Gravity (m/s²)

        # Aerodynamic coefficients (simplified)
        self.CL0 = 0.2            # Lift coefficient at zero AoA
        self.CLalpha = 4.47       # Lift curve slope
        self.CLq = 3.8            # Pitch rate contribution
        self.CLde = 0.43          # Elevator effectiveness

        self.CD0 = 0.03           # Parasitic drag
        self.CDalpha = 0.3        # Drag due to AoA

        self.CYbeta = -0.98       # Side force due to sideslip
        self.CYp = 0.0            # Side force due to roll rate
        self.CYr = 0.0            # Side force due to yaw rate
        self.CYda = 0.0           # Side force due to aileron
        self.CYdr = 0.17          # Side force due to rudder

        # Moment coefficients
        self.Clbeta = -0.13       # Roll moment due to sideslip
        self.Clp = -0.51          # Roll damping
        self.Clr = 0.25           # Roll moment due to yaw rate
        self.Clda = 0.156         # Aileron effectiveness
        self.Cldr = 0.0109        # Rudder effect on roll

        self.Cm0 = 0.0            # Pitch moment at zero AoA
        self.Cmalpha = -1.0       # Pitch moment due to AoA
        self.Cmq = -12.4          # Pitch damping
        self.Cmde = -1.28         # Elevator effectiveness

        self.Cnbeta = 0.073       # Yaw moment due to sideslip
        self.Cnp = -0.069         # Yaw moment due to roll rate
        self.Cnr = -0.095         # Yaw damping
        self.Cnda = -0.0012       # Aileron effect on yaw
        self.Cndr = -0.0657       # Rudder effectiveness

        # Engine (simplified)
        self.max_thrust = 75000.0  # N

        # Atmospheric properties (at 3000m)
        self.rho = 0.9093         # kg/m³

    def get_velocity_magnitude(self, state):
        """Total velocity magnitude"""
        u, v, w = state[0], state[1], state[2]
        return np.sqrt(u**2 + v**2 + w**2)

    def get_aerodynamic_angles(self, state):
        """Compute angle of attack and sideslip angle"""
        u, v, w = state[0], state[1], state[2]

        # Angle of attack
        alpha = np.arctan2(w, u)

        # Sideslip angle
        V = self.get_velocity_magnitude(state)
        beta = np.arcsin(np.clip(v / (V + 1e-6), -1, 1))

        return alpha, beta

    def get_load_factor(self, state, controls):
        """Compute load factor (g-force)"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = state
        delta_a, delta_e, delta_r, throttle = controls

        # Convert control inputs (normalized) to actual deflections
        delta_a_rad = np.deg2rad(delta_a * 21.5)
        delta_e_rad = np.deg2rad(delta_e * 25.0)
        delta_r_rad = np.deg2rad(delta_r * 30.0)

        V = self.get_velocity_magnitude(state)
        alpha, beta = self.get_aerodynamic_angles(state)
        Q = 0.5 * self.rho * V**2

        # Normalized rates
        q_hat = (q * self.c) / (2 * V + 1e-6)

        # Lift coefficient
        CL = self.CL0 + self.CLalpha * alpha + self.CLq * q_hat + self.CLde * delta_e_rad

        # Drag coefficient
        CD = self.CD0 + self.CDalpha * alpha**2

        # Aerodynamic forces
        L = Q * self.S * CL
        D = Q * self.S * CD

        # Transform to body frame
        Fz_aero = -D * np.sin(alpha) - L * np.cos(alpha)

        # Gravity component in body z-axis
        Fz_gravity = self.m * self.g * np.cos(theta) * np.cos(phi)

        # Total z-force
        Fz_total = Fz_aero + Fz_gravity

        # Load factor
        nz = -Fz_total / (self.m * self.g)

        return nz

    def derivatives(self, state, controls):
        """Compute state derivatives using RK4 integration"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = state
        delta_a, delta_e, delta_r, throttle = controls

        # Convert normalized controls to actual deflections
        delta_a_rad = np.deg2rad(delta_a * 21.5)
        delta_e_rad = np.deg2rad(delta_e * 25.0)
        delta_r_rad = np.deg2rad(delta_r * 30.0)

        # Aerodynamic angles
        V = self.get_velocity_magnitude(state)
        alpha, beta = self.get_aerodynamic_angles(state)

        # Dynamic pressure
        Q = 0.5 * self.rho * V**2

        # Normalized rates
        p_hat = (p * self.b) / (2 * V + 1e-6)
        q_hat = (q * self.c) / (2 * V + 1e-6)
        r_hat = (r * self.b) / (2 * V + 1e-6)

        # Force coefficients
        CL = self.CL0 + self.CLalpha * alpha + self.CLq * q_hat + self.CLde * delta_e_rad
        CD = self.CD0 + self.CDalpha * alpha**2
        CY = self.CYbeta * beta + self.CYp * p_hat + self.CYr * r_hat + \
             self.CYda * delta_a_rad + self.CYdr * delta_r_rad

        # Moment coefficients
        Cl = self.Clbeta * beta + self.Clp * p_hat + self.Clr * r_hat + \
             self.Clda * delta_a_rad + self.Cldr * delta_r_rad
        Cm = self.Cm0 + self.Cmalpha * alpha + self.Cmq * q_hat + self.Cmde * delta_e_rad
        Cn = self.Cnbeta * beta + self.Cnp * p_hat + self.Cnr * r_hat + \
             self.Cnda * delta_a_rad + self.Cndr * delta_r_rad

        # Aerodynamic forces (body frame)
        L_aero = Q * self.S * CL
        D_aero = Q * self.S * CD
        Y_aero = Q * self.S * CY

        Fx_aero = -D_aero * np.cos(alpha) + L_aero * np.sin(alpha)
        Fy_aero = Y_aero
        Fz_aero = -D_aero * np.sin(alpha) - L_aero * np.cos(alpha)

        # Thrust
        T = throttle * self.max_thrust
        Fx_thrust = T

        # Gravity (body frame)
        Fx_grav = -self.m * self.g * np.sin(theta)
        Fy_grav = self.m * self.g * np.cos(theta) * np.sin(phi)
        Fz_grav = self.m * self.g * np.cos(theta) * np.cos(phi)

        # Total forces
        Fx = Fx_aero + Fx_thrust + Fx_grav
        Fy = Fy_aero + Fy_grav
        Fz = Fz_aero + Fz_grav

        # Moments
        L_moment = Q * self.S * self.b * Cl
        M_moment = Q * self.S * self.c * Cm
        N_moment = Q * self.S * self.b * Cn

        # Translational accelerations
        u_dot = r * v - q * w + Fx / self.m
        v_dot = p * w - r * u + Fy / self.m
        w_dot = q * u - p * v + Fz / self.m

        # Rotational accelerations (simplified, ignoring Ixz coupling)
        p_dot = (self.Izz * L_moment + self.Ixz * N_moment -
                 (self.Izz * (self.Izz - self.Iyy) + self.Ixz**2) * q * r) / \
                (self.Ixx * self.Izz - self.Ixz**2)

        q_dot = (M_moment - (self.Ixx - self.Izz) * p * r - self.Ixz * (p**2 - r**2)) / self.Iyy

        r_dot = (self.Ixz * L_moment + self.Ixx * N_moment -
                 (self.Ixx * (self.Ixx - self.Iyy) + self.Ixz**2) * p * q) / \
                (self.Ixx * self.Izz - self.Ixz**2)

        # Euler angle rates
        phi_dot = p + q * np.sin(phi) * np.tan(theta) + r * np.cos(phi) * np.tan(theta)
        theta_dot = q * np.cos(phi) - r * np.sin(phi)
        psi_dot = (q * np.sin(phi) + r * np.cos(phi)) / (np.cos(theta) + 1e-6)

        # Position rates (NED frame)
        x_dot = (np.cos(theta) * np.cos(psi) * u +
                 (np.sin(phi) * np.sin(theta) * np.cos(psi) - np.cos(phi) * np.sin(psi)) * v +
                 (np.cos(phi) * np.sin(theta) * np.cos(psi) + np.sin(phi) * np.sin(psi)) * w)

        y_dot = (np.cos(theta) * np.sin(psi) * u +
                 (np.sin(phi) * np.sin(theta) * np.sin(psi) + np.cos(phi) * np.cos(psi)) * v +
                 (np.cos(phi) * np.sin(theta) * np.sin(psi) - np.sin(phi) * np.cos(psi)) * w)

        z_dot = (-np.sin(theta) * u + np.sin(phi) * np.cos(theta) * v +
                 np.cos(phi) * np.cos(theta) * w)

        return np.array([u_dot, v_dot, w_dot, p_dot, q_dot, r_dot,
                        phi_dot, theta_dot, psi_dot, x_dot, y_dot, z_dot])

    def step(self, state, controls, dt=0.01):
        """RK4 integration step"""
        k1 = self.derivatives(state, controls)
        k2 = self.derivatives(state + dt/2 * k1, controls)
        k3 = self.derivatives(state + dt/2 * k2, controls)
        k4 = self.derivatives(state + dt * k3, controls)

        new_state = state + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)

        # Wrap angles to [-π, π]
        new_state[6] = np.arctan2(np.sin(new_state[6]), np.cos(new_state[6]))  # phi
        new_state[7] = np.arctan2(np.sin(new_state[7]), np.cos(new_state[7]))  # theta
        new_state[8] = np.arctan2(np.sin(new_state[8]), np.cos(new_state[8]))  # psi

        return new_state

print("✓ Aircraft dynamics model loaded")

✓ Aircraft dynamics model loaded


## 🎮 Section 3: Gymnasium Environment with Enhanced Reward

In [ ]:
class Aircraft6DOFEnv(gym.Env):
    """6-DOF Aircraft Environment for 90° Heading Change"""

    def __init__(self, target_heading=90.0, max_steps=1000, dt=0.01):
        super().__init__()

        self.aircraft = AircraftDynamics()
        self.target_heading = np.deg2rad(target_heading)
        self.max_steps = max_steps
        self.dt = dt

        # Initial conditions
        self.initial_velocity = 200.0  # m/s
        self.initial_altitude = 3000.0  # m

        # Limits
        self.max_roll_angle = np.deg2rad(85)
        self.max_pitch_angle = np.deg2rad(60)
        self.max_roll_rate = np.deg2rad(180)
        self.max_pitch_rate = np.deg2rad(60)
        self.max_yaw_rate = np.deg2rad(60)
        self.min_velocity = 100.0
        self.max_velocity = 300.0
        self.altitude_tolerance = 50.0

        # Action space: [aileron, elevator, rudder, throttle]
        # All normalized to [-1, 1] except throttle [0, 1]
        self.action_space = gym.spaces.Box(
            low=np.array([-1.0, -1.0, -1.0, 0.0]),
            high=np.array([1.0, 1.0, 1.0, 1.0]),
            dtype=np.float32
        )

        # Observation space: [u, v, w, p, q, r, φ, θ, ψ, alt_err, V, nz, α]
        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(13,),
            dtype=np.float32
        )

        self.state = None
        self.step_count = 0
        self.prev_psi_error = np.pi / 2

    def _angle_difference(self, target, current):
        """Compute shortest angular distance"""
        diff = target - current
        return np.arctan2(np.sin(diff), np.cos(diff))

    def _get_observation(self):
        """Convert state to observation"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state

        # Altitude error
        altitude = -z
        altitude_error = altitude - self.initial_altitude

        # Velocity magnitude
        V = self.aircraft.get_velocity_magnitude(self.state)

        # Load factor (use zero controls for observation)
        nz = self.aircraft.get_load_factor(self.state, np.zeros(4))

        # Angle of attack
        alpha, _ = self.aircraft.get_aerodynamic_angles(self.state)

        # Normalize observation
        obs = np.array([
            u / 250.0,
            v / 50.0,
            w / 50.0,
            p / self.max_roll_rate,
            q / self.max_pitch_rate,
            r / self.max_yaw_rate,
            phi / self.max_roll_angle,
            theta / self.max_pitch_angle,
            self._angle_difference(self.target_heading, psi) / (np.pi/2),
            altitude_error / 100.0,
            V / 250.0,
            nz / 7.0,
            alpha / np.deg2rad(25)
        ], dtype=np.float32)

        return obs

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Initial state: straight and level flight at 200 m/s, heading 0°
        self.state = np.array([
            self.initial_velocity,  # u
            0.0,                    # v
            0.0,                    # w
            0.0,                    # p
            0.0,                    # q
            0.0,                    # r
            0.0,                    # phi
            0.0,                    # theta
            0.0,                    # psi (heading 0°)
            0.0,                    # x
            0.0,                    # y
            -self.initial_altitude  # z (NED frame)
        ])

        self.step_count = 0
        self.prev_psi_error = np.pi / 2

        return self._get_observation(), {}

    def compute_reward(self, state, action):
        """Enhanced reward function with all dynamics"""
        u, v, w, p, q, r, phi, theta, psi, x, y, z = state

        # ===== 1. HEADING PROGRESS (Primary) =====
        psi_error = abs(self._angle_difference(psi, self.target_heading))
        heading_progress = 1.0 - (psi_error / (np.pi/2))
        heading_reward = 500 * (np.exp(2 * heading_progress) - 1) / (np.exp(2) - 1)

        # Milestones
        milestone_bonus = 0
        if psi_error < np.deg2rad(45): milestone_bonus += 100
        if psi_error < np.deg2rad(20): milestone_bonus += 200
        if psi_error < np.deg2rad(10): milestone_bonus += 300
        if psi_error < np.deg2rad(5):  milestone_bonus += 500
        if psi_error < np.deg2rad(2):  milestone_bonus += 1000

        # ===== 2. ALTITUDE MAINTENANCE =====
        altitude = -z
        altitude_error = abs(altitude - self.initial_altitude)

        if altitude_error < self.altitude_tolerance:
            altitude_reward = 50 * (1 - (altitude_error / self.altitude_tolerance)**2)
        else:
            altitude_reward = -2 * (altitude_error - self.altitude_tolerance)

        # ===== 3. VELOCITY MAINTENANCE =====
        V = self.aircraft.get_velocity_magnitude(state)
        velocity_error = abs(V - self.initial_velocity)

        if velocity_error < 20:
            velocity_reward = 30 * (1 - velocity_error / 20)
        else:
            velocity_reward = -1.5 * velocity_error

        # ===== 4. COORDINATED FLIGHT =====
        beta = np.arctan2(v, u)
        sideslip_penalty = -50 * abs(beta)

        alpha = np.arctan2(w, u)
        alpha_deg = np.rad2deg(abs(alpha))
        if alpha_deg < 10:
            alpha_reward = 20
        elif alpha_deg < 15:
            alpha_reward = 10
        else:
            alpha_reward = -5 * (alpha_deg - 15)

        # ===== 5. LOAD FACTOR =====
        nz = self.aircraft.get_load_factor(state, action)

        if 1.0 <= nz <= 3.5:
            load_factor_reward = 30
        elif 3.5 < nz <= 5.0:
            load_factor_reward = 20 - 5 * (nz - 3.5)
        elif 5.0 < nz <= 7.0:
            load_factor_reward = -20 * (nz - 5.0)
        else:
            load_factor_reward = -100 * abs(nz - 5.0)

        if nz < 0.5:
            load_factor_reward = -200 * (0.5 - nz)

        # ===== 6. BANK ANGLE SHAPING =====
        phi_deg = np.rad2deg(abs(phi))

        if psi_error > np.deg2rad(10):
            optimal_bank = 50
            bank_error = abs(phi_deg - optimal_bank)
            bank_reward = 40 * np.exp(-bank_error / 15)
        else:
            bank_reward = 30 * (1 - phi_deg / 85)

        # ===== 7. TURN RATE =====
        if psi_error > np.deg2rad(5):
            angle_diff = self._angle_difference(self.target_heading, psi)
            desired_turn_sign = np.sign(angle_diff)
            actual_turn_sign = np.sign(r)

            if desired_turn_sign == actual_turn_sign:
                turn_rate_reward = 25 * min(abs(r) / np.deg2rad(20), 1.0)
            else:
                turn_rate_reward = -40 * abs(r) / np.deg2rad(20)
        else:
            turn_rate_reward = 15 * (1 - abs(r) / np.deg2rad(30))

        # ===== 8. SMOOTHNESS =====
        smoothness_penalty = (
            -10 * (abs(p) / self.max_roll_rate)**2 +
            -10 * (abs(q) / self.max_pitch_rate)**2 +
            -5 * (abs(r) / self.max_yaw_rate)**2
        )

        # ===== 9. PROGRESS TRACKING =====
        error_reduction = self.prev_psi_error - psi_error
        progress_reward = 100 * error_reduction / np.deg2rad(1)
        self.prev_psi_error = psi_error

        # ===== 10. TIME PENALTY =====
        time_penalty = -0.5 if psi_error > np.deg2rad(10) else -0.1

        # ===== 11. COMPLETION BONUS =====
        completion_bonus = 0
        if psi_error < np.deg2rad(2) and altitude_error < self.altitude_tolerance:
            completion_bonus = 3000
            if abs(p) < np.deg2rad(5) and abs(q) < np.deg2rad(5):
                completion_bonus += 1000
        elif psi_error < np.deg2rad(5) and altitude_error < self.altitude_tolerance * 1.5:
            completion_bonus = 1500
        elif psi_error < np.deg2rad(10) and altitude_error < self.altitude_tolerance * 2:
            completion_bonus = 500

        # ===== TOTAL REWARD =====
        total_reward = (
            heading_reward + milestone_bonus + altitude_reward + velocity_reward +
            sideslip_penalty + alpha_reward + load_factor_reward + bank_reward +
            turn_rate_reward + smoothness_penalty + progress_reward +
            time_penalty + completion_bonus
        )

        return total_reward

    def step(self, action):
        # Clip action to valid range
        action = np.clip(action, self.action_space.low, self.action_space.high)

        # Simulate dynamics
        self.state = self.aircraft.step(self.state, action, self.dt)
        self.step_count += 1

        # Compute reward
        reward = self.compute_reward(self.state, action)

        # Check termination conditions
        u, v, w, p, q, r, phi, theta, psi, x, y, z = self.state
        V = self.aircraft.get_velocity_magnitude(self.state)
        altitude = -z
        psi_error = abs(self._angle_difference(psi, self.target_heading))
        altitude_error = abs(altitude - self.initial_altitude)

        # Success condition
        success = psi_error < np.deg2rad(5) and altitude_error < self.altitude_tolerance

        # Failure conditions
        crashed = (
            altitude < 100 or
            V < self.min_velocity or
            V > self.max_velocity or
            abs(phi) > self.max_roll_angle or
            abs(theta) > self.max_pitch_angle
        )

        done = success or crashed or (self.step_count >= self.max_steps)
        truncated = self.step_count >= self.max_steps

        info = {
            'success': success,
            'crashed': crashed,
            'heading_error_deg': np.rad2deg(psi_error),
            'altitude_error': altitude_error,
            'velocity': V
        }

        return self._get_observation(), reward, done, truncated, info

print("✓ Environment created")

✓ Environment created


## 🧠 Section 4: SAC Neural Networks

In [ ]:
class GaussianPolicy(nn.Module):
    """Stochastic policy with state-dependent mean and std"""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()

        # Shared encoder
        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # Mean and log_std heads
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)

        # Action bounds (for proper scaling)
        self.register_buffer('action_scale', torch.FloatTensor([1.0, 1.0, 1.0, 0.5]))
        self.register_buffer('action_bias', torch.FloatTensor([0.0, 0.0, 0.0, 0.5]))

    def forward(self, state):
        features = self.encoder(state)
        mean = self.mean(features)
        log_std = self.log_std(features)
        log_std = torch.clamp(log_std, -20, 2)
        return mean, log_std

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()

        # Reparameterization trick
        normal = Normal(mean, std)
        x_t = normal.rsample()

        # Squash to [-1, 1] using tanh
        action = torch.tanh(x_t)

        # Scale to actual action range
        action_scaled = action * self.action_scale + self.action_bias

        # Compute log probability with tanh correction
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(self.action_scale * (1 - action.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)

        mean_scaled = torch.tanh(mean) * self.action_scale + self.action_bias

        return action_scaled, log_prob, mean_scaled


class QNetwork(nn.Module):
    """Twin Q-networks to prevent overestimation"""
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()

        # Q1 network
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        # Q2 network
        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, state, action):
        xu = torch.cat([state, action], 1)
        q1 = self.q1(xu)
        q2 = self.q2(xu)
        return q1, q2


class ReplayBuffer:
    """Experience replay buffer"""
    def __init__(self, state_dim, action_dim, max_size=1000000):
        self.max_size = max_size
        self.ptr = 0
        self.size = 0

        self.state = np.zeros((max_size, state_dim))
        self.action = np.zeros((max_size, action_dim))
        self.next_state = np.zeros((max_size, state_dim))
        self.reward = np.zeros((max_size, 1))
        self.done = np.zeros((max_size, 1))

    def add(self, state, action, next_state, reward, done):
        self.state[self.ptr] = state
        self.action[self.ptr] = action
        self.next_state[self.ptr] = next_state
        self.reward[self.ptr] = reward
        self.done[self.ptr] = done

        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample(self, batch_size):
        ind = np.random.randint(0, self.size, size=batch_size)

        return (
            torch.FloatTensor(self.state[ind]),
            torch.FloatTensor(self.action[ind]),
            torch.FloatTensor(self.next_state[ind]),
            torch.FloatTensor(self.reward[ind]),
            torch.FloatTensor(self.done[ind])
        )

print("✓ SAC networks defined")

✓ SAC networks defined


## 🤖 Section 5: SAC Agent

In [ ]:
class SAC:
    """Soft Actor-Critic Agent"""
    def __init__(
        self,
        state_dim,
        action_dim,
        hidden_dim=256,
        lr=3e-4,
        gamma=0.99,
        tau=0.005,
        alpha=0.2,
        automatic_entropy_tuning=True,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha

        # Policy network
        self.policy = GaussianPolicy(state_dim, action_dim, hidden_dim).to(device)
        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr)

        # Q-networks (twin)
        self.critic = QNetwork(state_dim, action_dim, hidden_dim).to(device)
        self.critic_target = QNetwork(state_dim, action_dim, hidden_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=lr)

        # Automatic entropy tuning
        self.automatic_entropy_tuning = automatic_entropy_tuning
        if automatic_entropy_tuning:
            self.target_entropy = -action_dim
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha_optimizer = torch.optim.Adam([self.log_alpha], lr=lr)

    def select_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)

        if evaluate:
            _, _, action = self.policy.sample(state)
        else:
            action, _, _ = self.policy.sample(state)

        return action.detach().cpu().numpy()[0]

    def update(self, replay_buffer, batch_size=256):
        # Sample batch
        state, action, next_state, reward, done = replay_buffer.sample(batch_size)
        state = state.to(self.device)
        action = action.to(self.device)
        next_state = next_state.to(self.device)
        reward = reward.to(self.device)
        done = done.to(self.device)

        # Update Q-functions
        with torch.no_grad():
            next_action, next_log_prob, _ = self.policy.sample(next_state)
            q1_next, q2_next = self.critic_target(next_state, next_action)
            q_next = torch.min(q1_next, q2_next) - self.alpha * next_log_prob
            q_target = reward + (1 - done) * self.gamma * q_next

        q1, q2 = self.critic(state, action)
        q1_loss = F.mse_loss(q1, q_target)
        q2_loss = F.mse_loss(q2, q_target)
        q_loss = q1_loss + q2_loss

        self.critic_optimizer.zero_grad()
        q_loss.backward()
        self.critic_optimizer.step()

        # Update policy
        new_action, log_prob, _ = self.policy.sample(state)
        q1_new, q2_new = self.critic(state, new_action)
        q_new = torch.min(q1_new, q2_new)

        policy_loss = (self.alpha * log_prob - q_new).mean()

        self.policy_optimizer.zero_grad()
        policy_loss.backward()
        self.policy_optimizer.step()

        # Update temperature (alpha)
        if self.automatic_entropy_tuning:
            alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()

            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()

            self.alpha = self.log_alpha.exp()

        # Soft update target networks
        for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        return {
            'q1_loss': q1_loss.item(),
            'q2_loss': q2_loss.item(),
            'policy_loss': policy_loss.item(),
            'alpha': self.alpha if isinstance(self.alpha, float) else self.alpha.item()
        }

    def save(self, filename):
        torch.save({
            'policy': self.policy.state_dict(),
            'critic': self.critic.state_dict(),
            'critic_target': self.critic_target.state_dict(),
        }, filename)

    def load(self, filename):
        checkpoint = torch.load(filename)
        self.policy.load_state_dict(checkpoint['policy'])
        self.critic.load_state_dict(checkpoint['critic'])
        self.critic_target.load_state_dict(checkpoint['critic_target'])

print("✓ SAC agent ready")

✓ SAC agent ready


## 🎓 Section 6: Training Configuration

In [ ]:
# Training hyperparameters
TOTAL_TIMESTEPS = 300000
START_TIMESTEPS = 10000  # Random exploration
BATCH_SIZE = 256
EVAL_FREQ = 5000
SAVE_FREQ = 25000

# SAC hyperparameters
HIDDEN_DIM = 256
LR = 3e-4
GAMMA = 0.99
TAU = 0.005
ALPHA = 0.2

# Environment
TARGET_HEADING = 90.0  # degrees
MAX_STEPS = 1000

print("Configuration:")
print(f"  Total timesteps: {TOTAL_TIMESTEPS:,}")
print(f"  Target heading: {TARGET_HEADING}°")
print(f"  Device: {device}")
print(f"  Batch size: {BATCH_SIZE}")

Configuration:
  Total timesteps: 300,000
  Target heading: 90.0°
  Device: cuda
  Batch size: 256


## 🚀 Section 7: Training Loop

In [ ]:
def evaluate_policy(agent, env, episodes=10):
    """Evaluate trained policy"""
    total_reward = 0
    successes = 0

    for _ in range(episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False

        while not done:
            action = agent.select_action(state, evaluate=True)
            state, reward, done, truncated, info = env.step(action)
            episode_reward += reward
            done = done or truncated

        total_reward += episode_reward
        if info.get('success', False):
            successes += 1

    return total_reward / episodes, successes / episodes


def train_sac():
    """Main training function"""

    # Create environment
    env = Aircraft6DOFEnv(target_heading=TARGET_HEADING, max_steps=MAX_STEPS)

    # Create agent
    agent = SAC(
        state_dim=13,
        action_dim=4,
        hidden_dim=HIDDEN_DIM,
        lr=LR,
        gamma=GAMMA,
        tau=TAU,
        alpha=ALPHA,
        automatic_entropy_tuning=True,
        device=device
    )

    # Replay buffer
    replay_buffer = ReplayBuffer(state_dim=13, action_dim=4, max_size=1000000)

    # Logging
    episode_rewards = []
    episode_lengths = []
    successes = []
    eval_rewards = []
    eval_success_rates = []
    timesteps_log = []

    # Training loop
    state, _ = env.reset()
    episode_reward = 0
    episode_length = 0
    episode_num = 0

    start_time = time.time()

    print("\n" + "="*70)
    print("Starting SAC Training for 90° Heading Change")
    print("="*70 + "\n")

    for t in range(TOTAL_TIMESTEPS):
        # Select action
        if t < START_TIMESTEPS:
            action = env.action_space.sample()
        else:
            action = agent.select_action(state, evaluate=False)

        # Step environment
        next_state, reward, done, truncated, info = env.step(action)
        episode_reward += reward
        episode_length += 1

        # Store transition
        replay_buffer.add(state, action, next_state, reward, done or truncated)

        state = next_state

        # Update agent
        if t >= START_TIMESTEPS:
            update_info = agent.update(replay_buffer, BATCH_SIZE)

        # Episode done
        if done or truncated:
            episode_rewards.append(episode_reward)
            episode_lengths.append(episode_length)
            successes.append(1 if info.get('success', False) else 0)

            episode_num += 1

            if episode_num % 10 == 0:
                print(f"Episode {episode_num:4d} | Steps: {t:6d} | "
                      f"Reward: {episode_reward:7.1f} | "
                      f"Heading Error: {info['heading_error_deg']:5.1f}° | "
                      f"Success: {info.get('success', False)}")

            state, _ = env.reset()
            episode_reward = 0
            episode_length = 0

        # Evaluation
        if t % EVAL_FREQ == 0 and t > START_TIMESTEPS:
            eval_reward, eval_success = evaluate_policy(agent, env, episodes=10)
            eval_rewards.append(eval_reward)
            eval_success_rates.append(eval_success)
            timesteps_log.append(t)

            elapsed = time.time() - start_time

            print("\n" + "="*70)
            print(f"Evaluation at timestep {t:,}")
            print(f"  Eval Reward: {eval_reward:7.1f}")
            print(f"  Success Rate: {eval_success*100:5.1f}%")
            print(f"  Avg Reward (last 100): {np.mean(episode_rewards[-100:]):7.1f}")
            print(f"  Success Rate (last 100): {np.mean(successes[-100:])*100:5.1f}%")
            print(f"  Time elapsed: {elapsed/60:.1f} min")
            print("="*70 + "\n")

        # Save model
        if t % SAVE_FREQ == 0 and t > START_TIMESTEPS:
            agent.save(f'sac_aircraft_90deg_step{t}.pth')
            print(f"✓ Model saved at timestep {t:,}")

    # Final save
    agent.save('sac_aircraft_90deg_final.pth')

    print("\n" + "="*70)
    print("Training Complete!")
    print("="*70)

    return agent, {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'successes': successes,
        'eval_rewards': eval_rewards,
        'eval_success_rates': eval_success_rates,
        'timesteps': timesteps_log
    }

# Run training
agent, training_data = train_sac()

/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(



Starting SAC Training for 90° Heading Change

Episode   10 | Steps:   8667 | Reward: -151609.8 | Heading Error:  82.7° | Success: False
Episode   20 | Steps:  12762 | Reward: -2035.0 | Heading Error:  87.7° | Success: False
Episode   30 | Steps:  13677 | Reward: -5376.0 | Heading Error:  90.1° | Success: False
Episode   40 | Steps:  14686 | Reward:  1452.8 | Heading Error:  88.6° | Success: False

Evaluation at timestep 15,000
  Eval Reward:  3694.2
  Success Rate:   0.0%
  Avg Reward (last 100): -55630.0
  Success Rate (last 100):   2.3%
  Time elapsed: 1.4 min

Episode   50 | Steps:  15437 | Reward:   747.6 | Heading Error:  90.6° | Success: False
Episode   60 | Steps:  16473 | Reward:  1182.6 | Heading Error:  85.2° | Success: False
Episode   70 | Steps:  17319 | Reward:  1750.8 | Heading Error:  88.4° | Success: False
Episode   80 | Steps:  18537 | Reward:  7933.5 | Heading Error:  87.6° | Success: False
Episode   90 | Steps:  19579 | Reward:  4061.9 | Heading Error:  84.8° | Succ

## 📊 Section 8: Visualization of Results

In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Smooth data for plotting
def smooth(data, window=50):
    return np.convolve(data, np.ones(window)/window, mode='valid')

# Episode rewards
axes[0, 0].plot(smooth(training_data['episode_rewards']))
axes[0, 0].set_title('Episode Rewards (Smoothed)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True, alpha=0.3)

# Success rate
success_rate = smooth(training_data['successes']) * 100
axes[0, 1].plot(success_rate)
axes[0, 1].set_title('Success Rate (Smoothed)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Success Rate (%)')
axes[0, 1].axhline(y=85, color='r', linestyle='--', label='Target (85%)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Evaluation rewards
axes[1, 0].plot(training_data['timesteps'], training_data['eval_rewards'], marker='o')
axes[1, 0].set_title('Evaluation Rewards', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Timesteps')
axes[1, 0].set_ylabel('Average Reward')
axes[1, 0].grid(True, alpha=0.3)

# Evaluation success rate
axes[1, 1].plot(training_data['timesteps'],
                np.array(training_data['eval_success_rates']) * 100,
                marker='o', color='green')
axes[1, 1].set_title('Evaluation Success Rate', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Timesteps')
axes[1, 1].set_ylabel('Success Rate (%)')
axes[1, 1].axhline(y=85, color='r', linestyle='--', label='Target (85%)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal Statistics:")
print(f"  Average Reward (last 100 episodes): {np.mean(training_data['episode_rewards'][-100:]):.1f}")
print(f"  Success Rate (last 100 episodes): {np.mean(training_data['successes'][-100:])*100:.1f}%")
print(f"  Final Eval Success Rate: {training_data['eval_success_rates'][-1]*100:.1f}%")

## 🎬 Section 9: Evaluate Trained Policy

In [ ]:
def visualize_trajectory(agent, env):
    """Visualize a single episode trajectory"""
    state, _ = env.reset()

    # Storage
    states = []
    actions = []
    rewards = []

    done = False
    while not done:
        action = agent.select_action(state, evaluate=True)
        states.append(env.state.copy())
        actions.append(action.copy())

        state, reward, done, truncated, info = env.step(action)
        rewards.append(reward)
        done = done or truncated

    states = np.array(states)
    actions = np.array(actions)
    rewards = np.array(rewards)

    # Create visualization
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    time_steps = np.arange(len(states)) * 0.01

    # Heading
    axes[0, 0].plot(time_steps, np.rad2deg(states[:, 8]), linewidth=2)
    axes[0, 0].axhline(y=90, color='r', linestyle='--', label='Target (90°)')
    axes[0, 0].set_title('Heading Angle', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('Heading (deg)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Altitude
    axes[0, 1].plot(time_steps, -states[:, 11], linewidth=2)
    axes[0, 1].axhline(y=3000, color='r', linestyle='--', label='Target (3000m)')
    axes[0, 1].fill_between(time_steps, 2950, 3050, alpha=0.2, color='green', label='Tolerance')
    axes[0, 1].set_title('Altitude', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Altitude (m)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Bank angle
    axes[1, 0].plot(time_steps, np.rad2deg(states[:, 6]), linewidth=2)
    axes[1, 0].axhline(y=85, color='r', linestyle='--', label='Limit')
    axes[1, 0].axhline(y=-85, color='r', linestyle='--')
    axes[1, 0].set_title('Bank Angle', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Bank (deg)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Velocity
    V = np.sqrt(states[:, 0]**2 + states[:, 1]**2 + states[:, 2]**2)
    axes[1, 1].plot(time_steps, V, linewidth=2)
    axes[1, 1].axhline(y=200, color='r', linestyle='--', label='Target (200 m/s)')
    axes[1, 1].set_title('Velocity', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Velocity (m/s)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Controls
    axes[2, 0].plot(time_steps, actions[:, 0], label='Aileron', linewidth=2)
    axes[2, 0].plot(time_steps, actions[:, 1], label='Elevator', linewidth=2)
    axes[2, 0].plot(time_steps, actions[:, 2], label='Rudder', linewidth=2)
    axes[2, 0].set_title('Control Deflections', fontsize=12, fontweight='bold')
    axes[2, 0].set_ylabel('Deflection (normalized)')
    axes[2, 0].set_xlabel('Time (s)')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)

    # Reward
    axes[2, 1].plot(time_steps, np.cumsum(rewards), linewidth=2, color='purple')
    axes[2, 1].set_title('Cumulative Reward', fontsize=12, fontweight='bold')
    axes[2, 1].set_ylabel('Cumulative Reward')
    axes[2, 1].set_xlabel('Time (s)')
    axes[2, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('trajectory.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print summary
    final_heading_error = abs(np.rad2deg(states[-1, 8]) - 90)
    final_altitude_error = abs(-states[-1, 11] - 3000)
    total_reward = np.sum(rewards)

    print("\nTrajectory Summary:")
    print(f"  Duration: {time_steps[-1]:.2f} seconds")
    print(f"  Final heading error: {final_heading_error:.2f}°")
    print(f"  Final altitude error: {final_altitude_error:.2f} m")
    print(f"  Total reward: {total_reward:.1f}")
    print(f"  Success: {info.get('success', False)}")

# Visualize 3 example trajectories
env_eval = Aircraft6DOFEnv(target_heading=90.0, max_steps=1000)

print("\n" + "="*70)
print("Visualizing Trained Policy")
print("="*70)

for i in range(3):
    print(f"\nTrajectory {i+1}:")
    visualize_trajectory(agent, env_eval)

## 💾 Section 10: Save and Download Model

In [ ]:
# Save final model
agent.save('sac_aircraft_90deg_final.pth')

# Download model (if running in Colab)
try:
    from google.colab import files
    files.download('sac_aircraft_90deg_final.pth')
    files.download('training_curves.png')
    files.download('trajectory.png')
    print("✓ Files downloaded successfully")
except:
    print("✓ Model saved locally as 'sac_aircraft_90deg_final.pth'")

print("\n" + "="*70)
print("Training Complete! 🎉")
print("="*70)
print("\nYour aircraft can now perform 90° heading changes!")
print("\nNext steps:")
print("  1. Test on different initial conditions")
print("  2. Try 180° heading changes")
print("  3. Add wind disturbances")
print("  4. Implement formation flying")